# Computational cost of alternating FC / FOSI spin-up segments

For a 2000-yr spin-up that alternates *b* years fully coupled (FC, B-case) with *g* years forced ocean–sea ice (FOSI, G-case),
each cell gives the total wallclock cost (fill and second label, months) and the saving relative to FC only (first label).
Costs come from the measured SYPD of each case, normalized to 104 nodes.

Outputs:
* `spinup_cost_saving_vs_FullCPL_<T>yr.pdf` (or `spinup_cost_total_<T>yr.pdf` with `plot_total_cost = True`), T = total spin-up length
* `data/computation/case_cost_summary.csv` — mean SYPD and cost of each case

**Setup** holds the paths; the **settings** cell holds the segment lengths and the measured SYPD/node counts of each case.

In [1]:
import os, sys
WF_ROOT = os.path.abspath("..")            # coupled_spinup_eval/
if WF_ROOT not in sys.path:
    sys.path.insert(0, WF_ROOT)
%matplotlib inline

import numpy as np

from util.computation_cost import AlternatingSimulationEvaluator, SimulationRun
from util.figures import publish

## Setup

In [2]:
# ---- paths ----
OUTPUT_ROOT  = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/v3spinup_paper"   # published figures/ and data/
OUTPUT_URL   = "https://web.lcrc.anl.gov/public/e3sm/diagnostic_output/ac.szhang/v3spinup_paper"
FIG_DIR      = f"{OUTPUT_ROOT}/figures/computation"
DATA_DIR     = f"{OUTPUT_ROOT}/data/computation"           # cached/derived data (index: data/README.md)


In [3]:
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
publish(OUTPUT_ROOT, FIG_DIR)
print("figures ->", FIG_DIR)
print("data    ->", DATA_DIR)
print("web     ->", FIG_DIR.replace(OUTPUT_ROOT, OUTPUT_URL))

figures -> /lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/v3spinup_paper/figures/computation
web     -> https://web.lcrc.anl.gov/public/e3sm/diagnostic_output/ac.szhang/v3spinup_paper/figures/computation


## 1 Simulation cost
### Settings

In [4]:
# Segment lengths (years) of FC (B) and FOSI (G), and total spin-up lengths
len_b = [0, 10, 20, 50, 100, 150, 200, 300, 500, 700, 1000, 1500, 2000]
len_g = [0, 10, 20, 60, 120, 160, 200, 400, 600, 800, 1000, 1500, 2000]
len_t = np.linspace(0, 2000, 501, dtype=int)  # 0 to 2000 years; the figure uses the last (2000)

# Measured throughput of each case (one entry per run segment)
sim_runs = [
    SimulationRun(
        code="E3SMv3-LR",
        exp="20231209.v3.LR.piControl-spinup.chrysalis",
        case_types=["B-Case"],
        nodes=[104] * 33 + [105, 105],
        sypd=[
            28.30, 26.00, None, 26.76, 28.11, 27.61, 27.50, None, 28.16, 28.33,
            28.33, 28.28, 28.37, 28.28, 28.26, 28.38, 28.44, 28.30, 28.42, 28.41,
            28.44, 28.25, 28.36, 28.40, 28.30, 28.37, 28.48, 28.37, 28.44, 28.39,
            28.29, 28.43, 28.31, None, 27.39
        ]
    ),
    SimulationRun(
        code="E3SMv3-LR",
        exp="20250509.v3.LR.GCTRL.TEST2.DATM.CPLHIST.chrysalis",
        case_types=["G-Case"],
        nodes=[54, 54],
        sypd=[34.44, 34.50]
    ),
    SimulationRun(
        code="E3SMv3-LR",
        exp="20250502.v3.LR.IGPRCPROG.DATM.CPLHIST.chrysalis",
        case_types=["IG-Case"],
        nodes=[54, 54],
        sypd=[30.52, 31.57]
    )
]

# Figure: False = fill total cost, label % saving vs FC only; True = fill/label total cost
plot_total_cost = False

### Compute & plot

In [5]:
evaluator = AlternatingSimulationEvaluator(len_b=len_b, len_g=len_g, len_t=len_t)

# Reference cost of each case
sypd_b, mpyr_b, cost_b = evaluator.calculate_cost(sim_runs[0].sypd, sim_runs[0].nodes)
sypd_g, mpyr_g, cost_g = evaluator.calculate_cost(sim_runs[1].sypd, sim_runs[1].nodes)
sypd_ig, mpyr_ig, cost_ig = evaluator.calculate_cost(sim_runs[2].sypd, sim_runs[2].nodes)

print("--- Simulation Summary ---")
print(f"B-Case:  SYPD = {sypd_b:.2f}, Months/yr = {mpyr_b:.5f}, Total = {cost_b:.2f} months")
print(f"G-Case:  SYPD = {sypd_g:.2f}, Months/yr = {mpyr_g:.5f}, Total = {cost_g:.2f} months")
print(f"IG-Case: SYPD = {sypd_ig:.2f}, Months/yr = {mpyr_ig:.5f}, Total = {cost_ig:.2f} months")

# Cost savings and total runtime for every (b, g, t)
evaluator.evaluate_costs_ratio_to_B(cost_b=cost_b, cost_g=cost_g)
evaluator.evaluate_costs_months(cost_b=mpyr_b, cost_g=mpyr_g)

# Summary table
summary_csv = os.path.join(DATA_DIR, "case_cost_summary.csv")
summary_df = evaluator.generate_case_summary(sim_runs)
summary_df.to_csv(summary_csv, index=False)
print("\nSaved summary to:", summary_csv)
print(summary_df)

cb_title = f"Total Cost of {len_t[-1]}-yr Ocean-Sea Ice Simulation (months)"
if plot_total_cost:
    out_pdf = os.path.join(FIG_DIR, f"spinup_cost_total_{len_t[-1]}yr.pdf")
    evaluator.plot_run_years_fill_cost_overlay(
        fig_name=out_pdf,
        fig_title="Overlay: Fill = Cost (months), Label = Cost for B+G Run",
        cb_title=cb_title,
    )
else:
    out_pdf = os.path.join(FIG_DIR, f"spinup_cost_saving_vs_FullCPL_{len_t[-1]}yr.pdf")
    evaluator.plot_run_years_fill_cost_savings_overlay(
        cost_b=mpyr_b,
        cost_g=mpyr_g,
        fig_name=out_pdf,
        fig_title="Overlay: Fill = Total Cost (months) \n Label= %Saving vs FC-only and Total wallclock time (months)",
        cb_title=cb_title,
    )

publish(FIG_DIR, DATA_DIR)
print("web:", out_pdf.replace(OUTPUT_ROOT, OUTPUT_URL))

--- Simulation Summary ---
B-Case:  SYPD = 28.14, Months/yr = 0.00117, Total = 2.34 months
G-Case:  SYPD = 34.47, Months/yr = 0.00049, Total = 0.99 months
IG-Case: SYPD = 31.05, Months/yr = 0.00055, Total = 1.10 months

Saved summary to: /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_spinup_paper/figure_data/computation/case_cost_summary.csv
        code                                                exp     type  \
0  E3SMv3-LR          20231209.v3.LR.piControl-spinup.chrysalis   B-Case   
1  E3SMv3-LR  20250509.v3.LR.GCTRL.TEST2.DATM.CPLHIST.chrysalis   G-Case   
2  E3SMv3-LR    20250502.v3.LR.IGPRCPROG.DATM.CPLHIST.chrysalis  IG-Case   

   avg_sypd  months_per_year  total_months  
0     28.14         0.001168          2.34  
1     34.47         0.000495          0.99  
2     31.04         0.000549          1.10  
web: https://web.lcrc.anl.gov/public/e3sm/diagnostic_output/ac.szhang/v3spinup_paper/figures/computation/spinup_cost_saving_vs_FullCPL_2000yr.pdf
